# 1. Environment Setup

## 1.1 Install and Import libraries

In [1]:
import sys
import subprocess
import pkgutil


# Required packages
required_packages = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'pyarrow': 'pyarrow',
    'pillow': 'pillow',
    'sklearn': 'scikit-learn',
    'tqdm': 'tqdm',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn', 
}


print("Checking and installing missing dependencies... \n")
missing = []
for module_name, package_name in required_packages.items():
    if pkgutil.find_loader(module_name) is None:
        missing.append(package_name)

if missing:
    print(f"Installing missing packages: {','.join(missing)}. \n")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
    print("Installation complete. \n")
else:
    print("All dependencies already installed.\n")

Checking and installing missing dependencies... 

Installing missing packages: pillow. 

Installation complete. 



In [3]:
# GPU Detection, Package Installation, and Library Verification
import os
import warnings

# Disable Hugging Face symlink warning
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
warnings.filterwarnings('ignore')

# GPU Detection
import torch
cuda_available = torch.cuda.is_available()
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU Detected: {gpu_name}")
    DEVICE = "cuda"
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 
                          'torch', 'torchvision', 'transformers'])
else:
    print("GPU not available. Using CPU")
    DEVICE = "cpu"
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 
                          'torch', 'torchvision', 'transformers'])

# Import Core Libraries
import time
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

from transformers import (
    DistilBertTokenizer,
    DistilBertModel
)

from torchvision import models, transforms
from PIL import Image

# Verify Installation
print("=" * 50)
print("Library Installation Verification")
print("=" * 50)

import torchvision
import transformers

print(f"\n✓ PyTorch Version: {torch.__version__}")
print(f"  CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  CUDA Version: {torch.version.cuda}")
    print(f"  GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print(f"  Running on: CPU")

print(f"\n✓ TorchVision Version: {torchvision.__version__}")
print(f"✓ Transformers Version: {transformers.__version__}")
print("\n" + "=" * 50)

GPU Detected: NVIDIA GeForce RTX 5070 Ti Laptop GPU
Library Installation Verification

✓ PyTorch Version: 2.12.0.dev20260402+cu128
  CUDA Available: True
  CUDA Version: 12.8
  GPU Device: NVIDIA GeForce RTX 5070 Ti Laptop GPU

✓ TorchVision Version: 0.27.0.dev20260402+cu128
✓ Transformers Version: 5.5.3



## 1.2 Configuration

In [4]:
DATA_PATH = "../data/processed/amazon_reviews_s30.parquet"
MULTIMODAL_DATA = "../data/multimodel/dataset_50k.parquet"
IMAGE_DIR = "../data/multimodel/images"
MODEL_DIR = "../models"
MODEL_SAVE_PATH = "../models/best_multimodal_model.pt"

SAMPLE_SIZE = 50000
RANDOM_STATE = 42

MAX_TEXT_LENGTH = 256
IMAGE_SIZE = 224

TEXT_IMAGE_PROJ_DIM = 256
METADATA_OUTPUT_DIM = 64

HELPFUL_PERCENTILE = 75

BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
TEXT_SPLIT_RATIO = 0.2

STEP_SIZE = 1
GAMMA = 0.9

## 1.3 Load and Create Dataset

- Create the label using HELPFUL_PERCENTILE
- Reduce the dataset to 50K for designing purpose of the model

In [6]:
df = pd.read_parquet(MULTIMODAL_DATA)
print("Original size:", len(df))

# Create new dataset for multimodel design
df = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)
print("Multimodal dataset created with size:", len(df))

Original size: 2826526
Multimodal dataset created with size: 50000


In [11]:
threshold = df["helpful_vote"].quantile(
    HELPFUL_PERCENTILE / 100
)
print(f"Labeling reviews as helpful if helpful_vote >= {threshold:.2f}")

df["label"] = (df["helpful_vote"] >= threshold).astype(int)
print(df["label"].value_counts())

Labeling reviews as helpful if helpful_vote >= 3.00
label
0    35126
1    14874
Name: count, dtype: int64


In [12]:
df.to_parquet(MULTIMODAL_DATA)
print("Multimodel dataset saved to:", MULTIMODAL_DATA)

Multimodel dataset saved to: ../data/multimodel/dataset_50k.parquet


# 2. Load the Dataset

In [5]:
# Text Features
TEXT_COLUMN = "review_text"

# Image Features
IMAGE_PATH_COLUMN = "image_path"

# Metadata Features
METADATA_COLUMNS = [
    "rating",
    "review_length",
    "is_verified",
    "image_count",
    "has_image",
    "days_since_first_review",
    "product_popularity",
]

# Target
LABEL_COLUMN = "label"

In [6]:
if not os.path.exists(MULTIMODAL_DATA):
    print(f"Error: File not found at {MULTIMODAL_DATA}")
    print("Please ensure the file exists before proceeding.")
else:
    df = pd.read_parquet(MULTIMODAL_DATA)
    print("Multimodal dataset loaded with size:", len(df))

Multimodal dataset loaded with size: 50000


In [7]:
# Validate columns
required_columns = [
    TEXT_COLUMN,
    IMAGE_PATH_COLUMN,
    LABEL_COLUMN,
    *METADATA_COLUMNS
]

missing = [c for c in required_columns if c not in df.columns]
assert len(missing) == 0, f"Missing columns: {missing}"

print("Dataset columns validated successfully.")

Dataset columns validated successfully.


In [8]:
df.iloc[25:28]

,rating,images,parent_asin,timestamp,helpful_vote,review_text,review_length,is_verified,image_count,has_image,days_since_first_review,product_popularity,label,image_url,image_path
2309935,5,"[{'attachment_type': 'IMAGE', 'large_image_url...",B08TLSB3WS,2021-03-29 17:52:26.476,14,"Pleasantly surprised! Wow, I was not expecting...",649,1,1,True,8897,11,1,https://images-na.ssl-images-amazon.com/images...,../data/multimodel/images\2309935.jpg
838367,5,[],B001OQC0JS,2010-03-30 12:11:27.000,2,So far so good.... I bought this camcorder bec...,583,1,0,False,4879,15,0,NaN,NaN
467409,5,[],B000H92BTM,2008-12-15 15:19:38.000,1,"Good Battery Good price, especially if you hav...",108,1,0,False,4409,45,0,NaN,NaN


In [9]:
df.dtypes

rating                               int8
images                             object
parent_asin                      category
timestamp                  datetime64[ms]
helpful_vote                        int32
review_text                           str
review_length                       int32
is_verified                          int8
image_count                         int16
has_image                            bool
days_since_first_review             int32
product_popularity                  int32
label                               int64
image_url                             str
image_path                            str
dtype: object

# 3. Create Split

In [10]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=TEXT_SPLIT_RATIO,
    stratify=df[LABEL_COLUMN],
    random_state=RANDOM_STATE
)

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

Train size: 40000, Test size: 10000


# 4. Build Model Components
INITIALIZE PREPROCESSORS

## 4.1 Text Tokenizer

In [11]:
tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

## 4.2 Image Transform Pipeline

In [12]:
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Why is this done???

# 5. Create Multimodal Dataset Class

In [13]:
class MultimodalDataset(Dataset):
    def __init__(self, dataframe, tokenizer, image_transform):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.image_transform = image_transform

        self.text_col = TEXT_COLUMN
        self.image_col = IMAGE_PATH_COLUMN
        self.metadata_cols = METADATA_COLUMNS
        self.label_col = LABEL_COLUMN

        self.max_text_len = MAX_TEXT_LENGTH

    
    def __len__(self):
        return len(self.df)

    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ------ Text Processing ------
        text = str(row[self.text_col])
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_text_len,
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # ------ Image Processing ------
        image_path = row[self.image_col]

        if isinstance(image_path, str) and os.path.exists(image_path):
            try:
                image = Image.open(image_path).convert("RGB")
                image = self.image_transform(image)
            except:
                image = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
        else:
            image = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)

        # ------ Metadata Processing ------
        metadata = torch.tensor(
            row[self.metadata_cols].values.astype(np.float32)
        )

        # ------ Label ------
        label = torch.tensor(row[self.label_col], dtype=torch.float32)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "image": image,
            "metadata": metadata,
            "label": label
        }

# 6. Create DataLoaders

In [14]:
train_dataset = MultimodalDataset(
    dataframe = train_df,
    tokenizer = tokenizer,
    image_transform = image_transform
)

test_dataset = MultimodalDataset(
    dataframe = test_df,
    tokenizer = tokenizer,
    image_transform = image_transform
)

In [15]:
# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# 7. Build Model Components

In [16]:
class MultimodalModel(nn.Module):

    def __init__(self):
        super().__init__()

        # ------ Text Branch -----
        self.text_encoder = DistilBertModel.from_pretrained(
            "distilbert-base-uncased"
        )
        self.text_proj = nn.Linear(768, TEXT_IMAGE_PROJ_DIM)

        # ------ Image Branch -----
        self.image_encoder = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )
        self.image_encoder.classifier = nn.Sequential()
        self.image_proj = nn.Linear(1280, TEXT_IMAGE_PROJ_DIM)

        # ------ Metadata Branch -----
        self.metadata_encoder = nn.Sequential(
            nn.Linear(len(METADATA_COLUMNS), METADATA_OUTPUT_DIM),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # ------ Fusion + Classifier -----
        self.classifier = nn.Sequential(
            nn.Linear( TEXT_IMAGE_PROJ_DIM * 2 + METADATA_OUTPUT_DIM, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )


    def forward(self, input_ids, attention_mask, image, metadata):
        # ------ Text Branch -----
        text_output = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_feat = text_output.last_hidden_state[:, 0]
        text_feat = self.text_proj(text_feat)

        # ------ Image Branch -----
        image_feat = self.image_encoder(image)
        image_feat = self.image_proj(image_feat)

        # ------ Metadata Branch -----
        metadata_feat = self.metadata_encoder(metadata)

        # ------ Fusion -----
        fused = torch.cat(
            [text_feat, image_feat, metadata_feat],
            dim=1
        )

        logits = self.classifier(fused)

        return logits.squeeze(1)

# 8. Training Setup

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultimodalModel().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=STEP_SIZE,
    gamma=GAMMA
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# 9. Training Loop

In [18]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for batch in loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        image = batch["image"].to(device)
        metadata = batch["metadata"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            image=image,
            metadata=metadata
        )

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    return total_loss / len(loader)

# 10. Validation Loop

In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

def evaluate(model, loader, device=device):
    model.eval()

    probs = []
    preds = []
    targets = []

    start_time = time.time()

    total_inference_time = 0
    total_samples = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            image = batch["image"].to(device)
            metadata = batch["metadata"].to(device)
            labels = batch["label"].to(device)

            # Inference time measurement
            infer_start = time.time()

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                image=image,
                metadata=metadata
            )

            infer_end = time.time()
            total_inference_time += (infer_end - infer_start)
            total_samples += labels.size(0)

            probabilities = torch.sigmoid(outputs)
            predictions = probabilities > 0.5

            probs.extend(probabilities.cpu().numpy())
            preds.extend(predictions.cpu().numpy())
            targets.extend(labels.cpu().numpy())

    total_eval_time = time.time() - start_time

    # ------- Metrics Calculation -------
    accuracy = accuracy_score(targets, preds)
    precision = precision_score(targets, preds)
    recall = recall_score(targets, preds)
    f1 = f1_score(targets, preds)

    roc_auc = roc_auc_score(targets, probs)
    conf_matrix = confusion_matrix(targets, preds)

    latency_per_sample = total_inference_time / total_samples

    results = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "ROC-AUC": roc_auc,
        "Confusion Matrix": conf_matrix,
        "Evaluation Time (sec)": total_eval_time,
        "Inference Latency (sec/sample)": latency_per_sample
    }

    return results

# 11. Model Checkpoints

In [20]:
best_acc = 0

for epoch in range(EPOCHS):
    train_start = time.time()
    train_loss = train_epoch(model, train_loader)
    training_time = time.time() - train_start

    val_results = evaluate(model, test_loader)
    val_acc = val_results["Accuracy"]

    scheduler.step()

    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f} - Val Acc: {val_acc:.4f} - Train Time: {training_time:.2f} sec")

    if val_acc > best_acc:
        best_acc = val_acc
        os.makedirs(MODEL_DIR, exist_ok=True) # Create model directory if it doesn't exist
        torch.save(model.state_dict(), MODEL_SAVE_PATH)

Epoch 1/3 - Train Loss: 12.7181 - Val Acc: 0.6946 - Train Time: 609.58 sec
Epoch 2/3 - Train Loss: 4.8502 - Val Acc: 0.7226 - Train Time: 535.81 sec
Epoch 3/3 - Train Loss: 2.1197 - Val Acc: 0.7213 - Train Time: 539.67 sec


# 12. Testing / Final Evaluation

In [21]:
model.load_state_dict(torch.load(MODEL_SAVE_PATH))

test_results = evaluate(model, test_loader)
test_acc = test_results["Accuracy"]

print(f"Final Test Accuracy: {test_acc:.4f}")
print(f"\nFull Evaluation Results:")
for metric, value in test_results.items():
    if metric != "Confusion Matrix":
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}:\n{value}")

Final Test Accuracy: 0.7226

Full Evaluation Results:
  Accuracy: 0.7226
  Precision: 0.5726
  Recall: 0.2666
  F1-score: 0.3638
  ROC-AUC: 0.6926
  Confusion Matrix:
[[6433  592]
 [2182  793]]
  Evaluation Time (sec): 72.0033
  Inference Latency (sec/sample): 0.0012


# 13. Inference Pipeline

In [22]:
# The function seems to be fishy
def predict(model, text, image_path, metadata_values, device=device):
    model.eval()
    
    # Text
    encoding = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=MAX_TEXT_LENGTH,
        return_tensors="pt"
    )
    
    # Image
    if not os.path.exists(image_path):
        print(f"Warning: Image not found at {image_path}, using placeholder")
        image = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
    else:
        try:
            image = Image.open(image_path).convert("RGB")
            image = image_transform(image).unsqueeze(0)
        except Exception as e:
            print(f"Warning: Could not load image: {e}, using placeholder")
            image = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
    
    # Metadata
    if len(metadata_values) != len(METADATA_COLUMNS):
        raise ValueError(f"Expected {len(METADATA_COLUMNS)} metadata values, got {len(metadata_values)}")
    
    metadata = torch.tensor(metadata_values, dtype=torch.float32).unsqueeze(0)

    with torch.no_grad():
        output = model(
            input_ids=encoding["input_ids"].to(device),
            attention_mask=encoding["attention_mask"].to(device),
            image=image.to(device),
            metadata=metadata.to(device)
        )
        prob = torch.sigmoid(output).item()
    
    return prob